# Vector Embeddings and FAISS Vector Stores

This notebook turns LangChain documentation into searchable vector stores: it loads a webpage, splits its content into overlapping chunks, embeds each chunk with OpenAI embedding models, and indexes the vectors in FAISS. It compares two embedding model sizes and two chunking configurations, saves each index locally, then retrieves the top three similar chunks for a natural-language query.

This demonstrates the ingestion and retrieval foundations of a RAG workflow. It does not generate an answer with a language model. Running the embedding cells requires a valid `OPENAI_API_KEY` in the environment or a local `.env` file.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Load LangChain's public document-loader documentation as the source corpus.
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/oss/python/integrations/document_loaders")
loaded_documents = loader.load()

C:\Users\vishe\AppData\Local\Temp\ipykernel_14652\716871301.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
# Reuse one pipeline to split source documents, embed each chunk, build a FAISS
# index, and save the resulting vector store for local reuse.
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

def convert_to_embeddings(chunk_size, chunk_overlap, documents, model, path):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    data_chunks = text_splitter.split_documents(documents)
    embeddings = OpenAIEmbeddings(model=model)
    vector_store = FAISS.from_documents(
        documents=data_chunks,
        embedding=embeddings
    )
    vector_store.save_local(path)
    print(f"created vector store with {len(data_chunks)} vectors saved to {path}")
    return vector_store

In [4]:
# Compare the small and large embedding models across two chunk sizes.
# The suffix _v2 identifies the smaller chunk-size configuration.
# Combination 1: small model, 1000-character chunks.
path = "./vector_store_1536_dim"
model = "text-embedding-3-small"
vector_store_1536_dim = convert_to_embeddings(chunk_size=1000, chunk_overlap=200, documents=loaded_documents, model=model, path=path)

# Combination 2: small model, 500-character chunks.
path = "./vector_store_1536_dim_v2"
model = "text-embedding-3-small"
vector_store_1536_dim_v2 = convert_to_embeddings(chunk_size=500, chunk_overlap=100, documents=loaded_documents, model=model, path=path)

# Combination 3: large model, 1000-character chunks.
path = "./vector_store_3072_dim"
model = "text-embedding-3-large"
vector_store_3072_dim = convert_to_embeddings(chunk_size=1000, chunk_overlap=200, documents=loaded_documents, model=model, path=path)

# Combination 4: large model, 500-character chunks.
path = "./vector_store_3072_dim_v2"
model = "text-embedding-3-large"
vector_store_3072_dim_v2 = convert_to_embeddings(chunk_size=500, chunk_overlap=100, documents=loaded_documents, model=model, path=path)

created vector store with 14 vectors saved to ./vector_store_1536_dim
created vector store with 22 vectors saved to ./vector_store_1536_dim_v2
created vector store with 14 vectors saved to ./vector_store_3072_dim
created vector store with 22 vectors saved to ./vector_store_3072_dim_v2


In [5]:
# Use one question to compare nearest-neighbor retrieval across all four indexes.
query = "What are the exact differences between how productivity tool loaders and PDF loaders handle data ingestion interfaces according to the documentation?"

In [13]:
# Retrieve the three most similar chunks; display the highest-ranked match.
result = vector_store_1536_dim.similarity_search(query, k=2)
print(result[0].page_content)
print("---------------------------------------------------------")
print(result[1].page_content)

​By category
​Productivity tools
The below document loaders allow you to load data from commonly used productivity tools.
Document LoaderAPI referenceAgentMailAgentMailLoaderGoogle ClassroomGoogleClassroomLoader
​Webpages
The below document loaders allow you to load webpages.
---------------------------------------------------------
Document LoaderDescriptionPackage/APIUnstructuredUses Unstructured’s open source library to load PDFsPackageUpstage Document Parse LoaderLoad PDF files using UpstageDocumentParseLoaderPackageDoclingLoad PDF files using DoclingPackagePyPDFLoaderLoad PDF files with pypdf (langchain-community)CommunityPyMuPDFLoaderLoad PDF files with PyMuPDF (langchain-community)CommunityPDFMinerLoaderLoad PDF files with PDFMiner (langchain-community)CommunityPDFPlumberLoaderLoad PDF files with pdfplumber (langchain-community)CommunityMinerULoad PDF and other documents using MinerUPackageUnDatasIOLoad PDF files using UnDatasIOPackageOpenDataLoader PDFLoad PDF files using OpenD

In [14]:
# Repeat the same query so differences reflect the embedding/chunking setup.
result = vector_store_1536_dim_v2.similarity_search(query, k=2)
print(result[0].page_content)
print("---------------------------------------------------------")
print(result[1].page_content)

​By category
​Productivity tools
The below document loaders allow you to load data from commonly used productivity tools.
Document LoaderAPI referenceAgentMailAgentMailLoaderGoogle ClassroomGoogleClassroomLoader
​Webpages
The below document loaders allow you to load webpages.
---------------------------------------------------------
Document LoaderDescriptionPackage/APIUnstructuredUses Unstructured’s open source library to load PDFsPackageUpstage Document Parse LoaderLoad PDF files using UpstageDocumentParseLoaderPackageDoclingLoad PDF files using DoclingPackagePyPDFLoaderLoad PDF files with pypdf (langchain-community)CommunityPyMuPDFLoaderLoad PDF files with PyMuPDF (langchain-community)CommunityPDFMinerLoaderLoad PDF files with PDFMiner (langchain-community)CommunityPDFPlumberLoaderLoad PDF files with pdfplumber


In [15]:
# Compare retrieval from the large embedding model with 1000-character chunks.
result = vector_store_3072_dim.similarity_search(query, k=2)
print(result[0].page_content)
print("---------------------------------------------------------")
print(result[1].page_content)

loadersOn this pageInterfaceBy categoryProductivity toolsWebpagesPDFsCloud providersCommon file typesAll document loadersIntegrations by componentDocument loader integrationsCopy pageCopy pageIntegrate with document loaders using LangChain Python.Copy pageCopy pageDocument loaders provide a standard interface for reading data from different sources (such as Slack, Notion, or Google Drive) into LangChain’s Document format.
---------------------------------------------------------
This ensures that data can be handled consistently regardless of the source.
All document loaders implement the BaseLoader interface.
Community document loaders are user-contributed and unverified. LangChain does not review or endorse these integrations; use them at your own risk.
​Interface
Each document loader may define its own parameters, but they share a common API:


In [16]:
# Compare retrieval from the large embedding model with 500-character chunks.
result = vector_store_3072_dim_v2.similarity_search(query, k=2)
print(result[0].page_content)
print("---------------------------------------------------------")
print(result[1].page_content)

splittersEmbedding modelsVector storesDocument loadersOn this pageInterfaceBy categoryProductivity toolsWebpagesPDFsCloud providersCommon file typesAll document loadersIntegrations by componentDocument loader integrationsCopy pageCopy pageIntegrate with document loaders using LangChain Python.Copy pageCopy pageDocument loaders provide a standard interface for reading data from different sources (such as Slack, Notion, or Google Drive) into LangChain’s Document format.
---------------------------------------------------------
This ensures that data can be handled consistently regardless of the source.
All document loaders implement the BaseLoader interface.
Community document loaders are user-contributed and unverified. LangChain does not review or endorse these integrations; use them at your own risk.
​Interface
Each document loader may define its own parameters, but they share a common API:
